# Q-Learning

Last time you were on Lake Ontario, but this time you have arrived on a [frozen lake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) of Simcoe trying to find another treasure.
As Lake Simcoe is ``less popular'' you found no information at all---you can only randomly explore and figure the best strategy.


### State space
```
MAP = [
    "FFFFFFFF",
    "FFFFFFFF",
    "FFGHFFFF",
    "FFFFFHFF",
    "FFFHFFFF",
    "FHHFFFHF",
    "FHFFHFHF",
    "FFFHFSFF",
]
```

The tile letters denote
- “S” for Start tile
- “G” for Goal tile
- “F” for frozen tile
- “H” for a tile with a hole

### Action space
```
LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
STAY = 4
```

### Transition
The player will move in intended direction with probability of 1/3 else will move in either perpendicular direction with equal probability of 1/3 in both directions.
When the player intends to stay, it stays in the current cell with probability of 1.

### Reward
The player receives +1 reward on staying at goal tile, -1 reward on staying at a hole tile, and +0 otherwise.

In [1]:
from grid_world import GridWorld

import numpy as np

In [2]:
env = GridWorld()

As you have no access to the transitions this time, you can only explore the environment through interactions.

In general, an environment provides two methods:
- `reset(seed) -> initial_state`: Starts a new trajectory
- `step(action) -> (next_state, reward)`: Transition from current timestep to the next timestep given `action`

In [3]:
map_action = {
    0: "L",
    1: "D",
    2: "R",
    3: "U",
    4: "S,"
}

def get_cell(idx):
    return (idx // env.nrow, idx % env.nrow)

def to_idx(row, col):
    return row * env.ncol + col

In [59]:
def q_learning(
    gamma: float,
    eps: float,
    alpha: float,
    max_iteration: int,
    seed: int = None,
    print_interval: int = 1000
):
    qf = np.zeros((env.num_states, env.num_actions))

    num_iterations = 0

    curr_state = env.reset(seed)
    sample_rng = np.random.RandomState(seed) if seed is not None else np.random
    while True:
        num_iterations += 1

        # ==================================================================
        # TODO: Implement Q-learning

        # Sample action and get (next_state, reward) pair

        # Update Q-function based on Bellman optimality equation

        # ==================================================================

        action = -1

        if sample_rng.uniform(0,1) > eps: # Exploit if greater than eps
            action = np.argmax(qf[curr_state])
        else: # Explore otherwise
            action = sample_rng.randint(0,5)
        next_state, reward = env.step(action)
        qf[curr_state][action] = qf[curr_state][action] + alpha*(reward + gamma*np.max(qf[next_state]) - qf[curr_state][action])
        curr_state = next_state
        if num_iterations % 200 == 0:
            curr_state = env.reset(seed)

        if num_iterations % print_interval == 0:
            print("ITER {} =========".format(num_iterations))
            print(np.max(qf, axis=-1).reshape((env.nrow, env.ncol)))
            print(np.array([map_action[action] for action in np.argmax(qf, axis=-1)]).reshape((env.nrow, env.ncol)))

        if num_iterations >= max_iteration:
            break

    return qf



In [60]:
alpha = 1e-3
eps = 0.5
gamma = 0.99
max_iteration = 1e5

In [61]:
qf_star = q_learning(alpha, eps, gamma, max_iteration)

ITER 1000 =========
[[ 1.24371430e-17  9.80973154e-10  1.00078882e-08  9.81069289e-14
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 9.50990050e-13  9.90988705e-09  9.90991992e-04  1.94242201e-16
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 1.96125616e-16  9.90990991e-04  1.00100100e+00 -9.99008010e-01
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 1.93307727e-17  9.81176050e-09  9.90990991e-06  9.81081081e-07
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 9.50990050e-19  0.00000000e+00  0.00000000e+00 -9.99999000e-01
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -1.00088010e+00 -1.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -1.00000000e+00  0.00000000e+00  0.00000000e+00
  -9.99999990e-01  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00

In [40]:
print(np.max(qf_star, axis=-1).reshape((env.nrow, env.ncol)))

[[ 1.06743683e-14  1.95264565e-11  9.90990000e-09  1.97235857e-15
   9.61749911e-15  1.40513054e-24  2.12264970e-28  1.12551553e-28]
 [ 9.90794775e-10  1.99188991e-08  9.91091091e-04  9.90892903e-07
   9.81178247e-14  9.61750842e-17  2.85611442e-23  1.88503366e-22]
 [ 9.81181151e-07  1.00090190e-03  1.00100100e+00 -9.99999900e-01
   9.52240085e-20  9.52135335e-20  1.98077653e-26  9.33092773e-30]
 [ 9.71371293e-10  9.81092011e-12  1.00090090e-03  4.88369493e-47
   9.90427340e-22 -1.00000000e+00  1.98914813e-31  1.01534187e-34]
 [ 1.16232263e-17  1.18471485e-16  0.00000000e+00 -1.00000000e+00
   1.04639461e-23  9.52512309e-34  9.43102661e-37  1.33905030e-37]
 [ 2.22927682e-22 -1.00000010e+00 -1.00000000e+00  0.00000000e+00
   9.71276800e-32  9.42893810e-31 -1.00000000e+00  9.05200593e-40]
 [ 2.23225091e-29 -1.00000000e+00  0.00000000e+00  0.00000000e+00
  -1.00000000e+00  0.00000000e+00 -1.00000000e+00  8.96328722e-42]
 [ 1.06880734e-28  5.98932143e-46  1.38709206e-72 -1.00000000e+00
   

In [41]:
pi_star = np.argmax(qf_star, axis=-1)
print(np.array([map_action[action] for action in pi_star]).reshape((env.nrow, env.ncol)))

[['L' 'L' 'D' 'R' 'L' 'L' 'D' 'D']
 ['R' 'D' 'L' 'U' 'D' 'L' 'D' 'U']
 ['R' 'R' 'S,' 'U' 'S,' 'U' 'U' 'D']
 ['R' 'S,' 'U' 'S,' 'L' 'L' 'R' 'U']
 ['D' 'S,' 'S,' 'L' 'R' 'D' 'U' 'U']
 ['L' 'U' 'R' 'S,' 'U' 'L' 'U' 'S,']
 ['L' 'D' 'S,' 'S,' 'L' 'S,' 'D' 'R']
 ['R' 'D' 'L' 'U' 'S,' 'R' 'D' 'S,']]


In [ ]:
env.desc